In [1]:
# Importando bibliotecas
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from pandas.tseries.offsets import MonthEnd
from functions import *
from FUNCTIONS_LTP import *
import locale
from openpyxl import load_workbook
from openpyxl.utils import range_boundaries
import tempfile
import shutil
import math
from collections import defaultdict
import pyarrow

locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')  # Para Windows
timer = Temporizador()
timer.iniciar()

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Aumenta o limite de largura da coluna para exibição
pd.set_option('display.expand_frame_repr', False)

# Detecta se o script está sendo executado de um .py ou de um notebook
try:
    caminho_base = Path(__file__).resolve().parent
except NameError:
    # __file__ não existe em Jupyter ou ambiente interativo
    caminho_base = Path.cwd()
    
pasta_input = caminho_base.parent / '01_INPUT'
pasta_output = caminho_base.parent / '02_OUTPUT'
pasta_painel = caminho_base.parent / '03_EXCEL'

# Carregar bd_mat_param uma vez
bd_mat_param = pd.read_excel(
    pasta_input / 'matriz_parametros.xlsx',
    sheet_name='matriz_parametros',
    engine='calamine',
    dtype={'produto': str}
)
bd_mat_param['produto'] = bd_mat_param['produto'].astype(str).str.strip().str.upper()
produtos_param = set(bd_mat_param['produto'])
colunas_manter = [
    'produto', 'unidade_fat', 'unidade_prod', 'tipo_abast', 'prioridade'
]

bd_mat_param = bd_mat_param[colunas_manter].reset_index(drop=True)

# Carregar base_dados_produtos
bd_prod = pd.read_excel(
    pasta_input / 'base_dados_produtos.xlsx',
    sheet_name='base_dados_produtos',
    engine='calamine',
    dtype={'cod_produto': str}
)
bd_prod['cod_produto'] = bd_prod['cod_produto'].astype(str).str.strip().str.upper()
bd_prod['cod_produto'] = bd_prod['cod_produto'].astype(str).str.strip().str.upper()
bd_prod = bd_prod[bd_prod['cod_produto'].isin(produtos_param)].copy().reset_index(drop=True)
if 'mes_ref' in bd_prod.columns:
    bd_prod['mes_ref'] = pd.to_datetime(bd_prod['mes_ref'], errors='coerce', dayfirst=True)

colunas_manter = ['mes_ref', 'empresa', 'cod_produto', 'descricao', 'linha_prod', 'familia_prod', 'tipo_produto', 'curva_abc', 'curva_123', 'estoq_seg_pcs', 'estoq_seg_kg', 'estoq_inicial_pcs', 'estoq_inicial_kg', 'carteira_arraste_mes_anterior', 'carteira_mes_atual', 'previsao_pcs', 'saldo_previsao_pcs', 'peso_produto_kg', 'estoq_transf_pcs', 'lote_econ', 'qtd_emb', 'lote_min']
bd_prod = bd_prod[colunas_manter].reset_index(drop=True)

# Carregar estruturas
colunas_bd_estruturas = ['mes_ref', 'empresa', 'cod_prod_acabado', 'cod_insumo', 'descricao', 'qtd_utilizada_pcs']
bd_estruturas = pd.read_excel(
    pasta_input / 'estruturas.xlsx',
    sheet_name='estruturas',
    engine='calamine',
    dtype={'cod_prod_acabado': str}
)
bd_estruturas['cod_prod_acabado'] = bd_estruturas['cod_prod_acabado'].astype(str).str.strip().str.upper()
bd_estruturas['cod_insumo'] = bd_estruturas['cod_insumo'].astype(str).str.strip().str.upper()
bd_estruturas = bd_estruturas[bd_estruturas['cod_prod_acabado'].isin(produtos_param)].copy().reset_index(drop=True)
bd_estruturas = bd_estruturas[colunas_bd_estruturas]
# Elminar coluna mes_ref e remover duplicatas
bd_estruturas = bd_estruturas.drop(columns=['mes_ref'])
bd_estruturas = bd_estruturas.drop_duplicates(subset=['empresa', 'cod_prod_acabado', 'cod_insumo']).reset_index(drop=True)

print("✅ Mapeamento de pastas e importação de tabelas concluído!")

✅ Mapeamento de pastas e importação de tabelas concluído!


In [2]:
# Gerando a tabela Calendário

# Definir intervalo de datas: do primeiro mês ao último mês completo
data_inicial = bd_prod['mes_ref'].min().replace(day=1)
data_final = bd_prod['mes_ref'].max() + MonthEnd(1)

# Gerar datas diariamente entre data_inicial e data_final
datas = pd.date_range(start=data_inicial, end=data_final, freq='D')

# Mapeamento dos dias da semana
dias_semana = {0: 'SEG', 1: 'TER', 2: 'QUA', 3: 'QUI', 4: 'SEX', 5: 'SÁB', 6: 'DOM'}

# Construção do DataFrame
df_calendario = pd.DataFrame({
    'data_calend': datas,
    'mes_calend': datas.to_series().apply(lambda d: d.replace(day=1)),
    'dia_calend': datas.to_series().dt.weekday.map(dias_semana)
})

# Importar aba Dia_Semana da planilha dados_calendario, com dados por dia da semana x Unidade
df_dia_semana = pd.read_excel(pasta_input / 'dados_calendario.xlsx', engine='calamine', sheet_name='Dia_Semana')

# Normalização (unpivot)
bd_dia_semana = df_dia_semana.melt(
    id_vars=['UNIDADE'],
    var_name='DIA_DE_SEMANA',
    value_name='OCUPACAO'
)

# Remove linhas onde OCUPACAO está vazia ou inválida (ex: '-')
bd_dia_semana = bd_dia_semana[bd_dia_semana['OCUPACAO'].notna()]
bd_dia_semana = bd_dia_semana[bd_dia_semana['OCUPACAO'] != '-']
bd_dia_semana['OCUPACAO'] = bd_dia_semana['OCUPACAO'].astype(float)

# Fazendo o merge entre o calendário e os dados de ocupação por unidade/dia da semana
df_calendario_ocup = df_calendario.merge(
    bd_dia_semana,
    left_on='dia_calend',
    right_on='DIA_DE_SEMANA',
    how='left'
)

# Selecionar apenas as colunas desejadas
df_calendario_ocup = df_calendario_ocup[
    ['data_calend', 'mes_calend', 'dia_calend', 'UNIDADE', 'OCUPACAO']
]

# Renomear colunas para maiúsculas
df_calendario_ocup.columns = [col.upper() for col in df_calendario_ocup.columns]

# Importar aba Feriados da planilha dados_calendario, com dados por dia da semana x Unidade
df_feriados = pd.read_excel(pasta_input / 'dados_calendario.xlsx', engine='calamine', sheet_name='Feriados')

# Garantir que as datas estejam no mesmo formato
df_feriados['FERIADO'] = pd.to_datetime(df_feriados['FERIADO'], format="%d/%m/%Y")

# Realizar o merge
df_calendario_ocup = df_calendario_ocup.merge(
    df_feriados[['FERIADO', 'UNIDADE', 'TIPO', 'NORMAL', 'REVEZAMENTO']],
    left_on=['DATA_CALEND', 'UNIDADE'],
    right_on=['FERIADO', 'UNIDADE'],
    how='left'
)

data_agora = datetime.now()
# Inserindo coluna data e hora agora
df_calendario_ocup['DATA_AGORA'] = data_agora

# NOVO_NORMAL: Se TIPO for NaN, retorna OCUPACAO; senão, retorna TIPO NORMAL
df_calendario_ocup['NOVO_NORMAL'] = df_calendario_ocup.apply(lambda row: row['OCUPACAO'] if pd.isna(row['TIPO']) else row['NORMAL'],axis=1)

# NOVO_REVEZAMENTO: Se TIPO for NaN, retorna 1; senão, retorna TIPO REVEZAMENTO
df_calendario_ocup['NOVO_REVEZAMENTO'] = df_calendario_ocup.apply(lambda row: 1 if pd.isna(row['TIPO']) else row['REVEZAMENTO'],axis=1)

# Coluna DIAS_NOR_CALEND: se DATA_CALEND <= DATA_AGORA retorna 0, senão NOVO_NORMAL
df_calendario_ocup['DIAS_NOR_CALEND_'] = df_calendario_ocup.apply(lambda row: 0 if row['DATA_CALEND'] <= row['DATA_AGORA'] else row['NOVO_NORMAL'],axis=1)

# Coluna DIAS_REV_CALEND: se DATA_CALEND <= DATA_AGORA retorna 0, senão NOVO_REVEZAMENTO
df_calendario_ocup['DIAS_REV_CALEND_'] = df_calendario_ocup.apply(lambda row: 0 if row['DATA_CALEND'] <= row['DATA_AGORA'] else row['NOVO_REVEZAMENTO'],axis=1)

# Coluna PARCIAL_HOJE: se DATA_CALEND = DATA_AGORA, RETORNA 1 - (Hora Agora / 24)
hora_agora = data_agora.hour + data_agora.minute / 60 + data_agora.second / 3600
data_hoje = data_agora.date()

df_calendario_ocup['PARCIAL_HOJE'] = df_calendario_ocup['DATA_CALEND'].dt.date.apply(lambda d: 1 - (hora_agora / 24) if d == data_hoje else 0)

df_calendario_ocup['DIAS_NOR_CALEND'] = df_calendario_ocup['DIAS_NOR_CALEND_'] + df_calendario_ocup['PARCIAL_HOJE']
df_calendario_ocup['DIAS_REV_CALEND'] = df_calendario_ocup['DIAS_REV_CALEND_'] + df_calendario_ocup['PARCIAL_HOJE']

bd_calend = df_calendario_ocup.groupby(['MES_CALEND', 'UNIDADE'], as_index=False)[['DIAS_NOR_CALEND', 'DIAS_REV_CALEND']].sum()
bd_calend = bd_calend.sort_values(by=['MES_CALEND', 'UNIDADE'])
bd_calend = bd_calend.rename(columns={
    'DIAS_NOR_CALEND': 'TOT_DIAS_NOR_CALEND',
    'DIAS_REV_CALEND': 'TOT_DIAS_REV_CALEND'
})

bd_calend['TOT_HORAS_NOR_CALEND'] = bd_calend['TOT_DIAS_NOR_CALEND'] * 24
bd_calend['TOT_HORAS_REV_CALEND'] = bd_calend['TOT_DIAS_REV_CALEND'] * 24

print("✅ Geração de tabelas de calendário concluído!")

✅ Geração de tabelas de calendário concluído!


In [3]:
# Inicio Step = GERAR DADOS LTP
bd_prod['NEC_BASE_PROD_PCS'] = (
    bd_prod['estoq_inicial_pcs'] 
    - bd_prod['carteira_mes_atual'] 
    - bd_prod['carteira_arraste_mes_anterior'] 
    - bd_prod['previsao_pcs'] 
    - bd_prod['estoq_seg_pcs']
)

bd_prod['PRIORIDADE'] = 1

bd_mat_param_prioridade_1 = bd_mat_param[bd_mat_param['prioridade'] == 1]

# Join completo entre bd_prod e bd_mat_param
bd_prod_nec = pd.merge(
    bd_mat_param_prioridade_1,
    bd_prod,
    left_on=['produto', 'unidade_fat', 'prioridade'],
    right_on=['cod_produto', 'empresa', 'PRIORIDADE'],
    how='outer'  # FULL OUTER JOIN
)

# Importar aba matriz_regioes da planilha matriz regioes
bd_matriz_regioes = pd.read_excel(pasta_input / 'matriz_regioes.xlsx', sheet_name='matriz_regioes', engine='calamine')

# Merge para adicionar a Região  de Faturamento na base de produtos
bd_prod_nec = bd_prod_nec.merge(
    bd_matriz_regioes,
    how='left',
    left_on='empresa',
    right_on='Unidade'
)

bd_prod_nec = bd_prod_nec.rename(columns={'Unidade': 'Unidade_Fat'})
bd_prod_nec = bd_prod_nec.rename(columns={'Região': 'Reg_Unid_Fat'})

# Merge para adicionar a Região Unidade Produtiva na base de produtos
bd_prod_nec = bd_prod_nec.merge(
    bd_matriz_regioes,
    how='left',
    left_on='unidade_prod',
    right_on='Unidade'
)

bd_prod_nec = bd_prod_nec.rename(columns={'Unidade': 'Unidade_Prod'})
bd_prod_nec = bd_prod_nec.rename(columns={'Região': 'Reg_Unid_Prod'})

# Criando a coluna MESMA REGIAO, com os critérios aplicados
bd_prod_nec['MESMA_REG'] = bd_prod_nec.apply(
    lambda row: "NAO" if pd.isna(row['Reg_Unid_Prod']) else ("SIM" if row['Reg_Unid_Fat'] == row['Reg_Unid_Prod'] else "NAO"),
    axis=1
)

# Inserindo a coluna Mes_Ref_Ant na bd_prod, e criando uma bd_prod_mes_anterior
bd_prod_mes_anterior = bd_prod[['mes_ref', 'empresa', 'cod_produto', 'descricao', 'saldo_previsao_pcs']].copy()
bd_prod_mes_anterior = bd_prod_mes_anterior.rename(columns=
    {'mes_ref': 'MES_REF', 
     'empresa': 'EMPRESA', 
     'cod_produto': 'COD_PROD', 
     'descricao': 'DESC_PROD',
     'saldo_previsao_pcs': 'SALDO_PREV_PROX_MES_PCS'
     }
)

# Criando coluna MES_REF_ANT para definir mes do saldo da previsão
bd_prod_mes_anterior['MES_REF_ANT'] = (bd_prod_mes_anterior['MES_REF'] - pd.DateOffset(months=1)).dt.to_period('M').dt.to_timestamp()

# Adicionar a coluna SALDO_PREV_PROX_MES_PCS na bd_prod_nec
# Criando um dicionário com as chaves de busca e a coluna que você quer trazer
map_dict = bd_prod_mes_anterior.set_index(['MES_REF_ANT', 'EMPRESA', 'COD_PROD'])['SALDO_PREV_PROX_MES_PCS'].to_dict()

# Usando map para trazer a coluna 'SALDO_PREV_PROX_MES_PCS' para bd_prod_nec
bd_prod_nec['SALDO_PREV_PROX_MES_PCS'] = bd_prod_nec[['mes_ref', 'empresa', 'cod_produto']].apply(
    lambda row: map_dict.get((row['mes_ref'], row['empresa'], row['cod_produto']), None), axis=1
)

# Eliminando linhas filtrando mes_ref que não seja null
bd_prod_nec = bd_prod_nec[bd_prod_nec['mes_ref'].notna()]

# Criando a coluna PCS_NEC_PROD_MESMA_REG_SIM_NAO
# Preencher todos os NaN/None das colunas usadas no cálculo com zero
colunas_nec = [
    'carteira_arraste_mes_anterior', 'carteira_mes_atual', 'saldo_previsao_pcs',
    'SALDO_PREV_PROX_MES_PCS', 'estoq_seg_pcs', 'estoq_inicial_pcs', 'estoq_transf_pcs'
]
for col in colunas_nec:
    bd_prod_nec[col] = pd.to_numeric(bd_prod_nec[col], errors='coerce').fillna(0.0)

# Agora pode aplicar o cálculo normalmente
bd_prod_nec['PCS_NEC_PROD_MESMA_REG_SIM_NAO'] = bd_prod_nec.apply(
    lambda row: (
        row['carteira_arraste_mes_anterior'] + row['carteira_mes_atual'] + row['saldo_previsao_pcs'] + row['SALDO_PREV_PROX_MES_PCS'] + row['estoq_seg_pcs']
        - (row['estoq_inicial_pcs'] + row['estoq_transf_pcs'])
    ) if row['tipo_produto'] == 'MR'
    else (
        row['carteira_arraste_mes_anterior'] + row['carteira_mes_atual'] + row['saldo_previsao_pcs'] + row['estoq_seg_pcs']
        - (row['estoq_inicial_pcs'] + row['estoq_transf_pcs'])
    ),
    axis=1
)

# Aplicar substituir null por Zero na coluna SALDO_PREV_PROX_MES_PCS
bd_prod_nec['SALDO_PREV_PROX_MES_PCS'] = bd_prod_nec['SALDO_PREV_PROX_MES_PCS'].fillna(0)

# Padronizando nomes de colunas para concluir bd_prod_nec
bd_prod_nec = bd_prod_nec.rename(columns=
    {'mes_ref': 'MES_REF', 
     'empresa': 'EMPRESA', 
     'cod_produto': 'COD_PROD', 
     'descricao': 'DESC_PROD',
     'linha_prod': 'LINHA_PROD',
     'familia_prod': 'FAMILIA_PROD',
     'tipo_produto': 'TIPO_PROD',
     'curva_abc': 'CURVA_ABC',
     'curva_123': 'CURVA_123',
     'estoq_seg_pcs': 'EST_SEG_PCS',
     'estoq_inicial_pcs': 'EST_INI_PCS',
     'carteira_arraste_mes_anterior': 'CART_ARR_MES_ANT',
     'carteira_mes_atual': 'CART_MES_ATUAL',
     'previsao_pcs': 'PREV_PCS',
     'saldo_previsao_pcs': 'SALDO_PREV_PCS',
     'peso_produto_kg': 'PESO_PROD_KG',
     'estoq_transf_pcs': 'EST_TRANS_PCS',
     'unidade_prod': 'UNID_PROD',
     'MESMA_REG': 'MESMA_REG',
     'saldo_previsao_pcs': 'SALDO_PREV_PCS',
     'lote_econ': 'LOTE_ECON',
     'qtd_emb': 'QTD_EMB',
     'lote_min': 'LOTE_MIN',
     }
)

# Definindo colunas e encerrando processo de formação da bd_prod_nec
bd_prod_nec = bd_prod_nec[
    ['MES_REF', 'EMPRESA', 'UNID_PROD', 'MESMA_REG', 'COD_PROD', 'DESC_PROD', 
     'LINHA_PROD', 'FAMILIA_PROD', 'TIPO_PROD', 'CURVA_ABC','CURVA_123', 'LOTE_ECON','QTD_EMB', 'LOTE_MIN', 
     'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'PREV_PCS', 
     'SALDO_PREV_PCS', 'PESO_PROD_KG', 'EST_TRANS_PCS','PCS_NEC_PROD_MESMA_REG_SIM_NAO',
     'SALDO_PREV_PROX_MES_PCS']
].reset_index(drop=True)

In [4]:
### Transformação 04_GerarBaseLTP para gerar a base LTP

# Carregar base_dados_roteiros
bd_rot = pd.read_excel(
    pasta_input / 'base_dados_roteiros.xlsx',
    sheet_name='base_dados_roteiros',
    dtype={'cod_produto': str}
)

bd_rot['cod_produto'] = bd_rot['cod_produto'].astype(str).str.strip().str.upper()
bd_rot = bd_rot[bd_rot['cod_produto'].isin(produtos_param)].copy().reset_index(drop=True)

if 'mes_ref' in bd_rot.columns:
    bd_rot['mes_ref'] = pd.to_datetime(bd_rot['mes_ref'], errors='coerce', dayfirst=True)
    
colunas_manter = ['mes_ref', 'empresa', 'cod_produto', 'descricao', 'linha_prod', 'familia_prod', 'tipo_produto', 'cod_ferramenta', 'n_cavidade', 'ciclo_molde', 'pcs_hr', 'mo', 'alocacao_recurso', 'grupo_setor', 'prioridade']

bd_rot = bd_rot[colunas_manter].reset_index(drop=True)

# Adicionando nova coluna na tabela de roteiros
bd_rot['COD_FERR'] = (bd_rot['cod_ferramenta'] + "|" + bd_rot['empresa'].str[:3]).str.upper()

# Faz o merge trazendo somente as 4 colunas desejadas, associando pelos campos de chave
bd_rot = bd_rot.merge(
    bd_calend, 
    how='left', 
    left_on=['mes_ref', 'empresa'], 
    right_on=['MES_CALEND', 'UNIDADE']
)

# Removendo as colunas que não quero na bd_rot
colunas_excluir = ['MES_CALEND', 'UNIDADE']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Carregando o Calendario de Recursos
bd_calend_rec = pd.read_excel(pasta_input / 'calend_recursos.xlsx', sheet_name='calend_recursos')

# Inserindo coluna PER_TURNO no calendário de recursos
bd_calend_rec['PER_TURNO'] = bd_calend_rec['turno_maq'] / 3

# Inserindo colunas do calendário de recursos na bd_rot
bd_rot = bd_rot.merge(
    bd_calend_rec, 
    how='left', 
    left_on=['mes_ref', 'empresa', 'alocacao_recurso'], 
    right_on=['mes_ref', 'unidade', 'maq']
)

colunas_excluir = ['unidade', 'maq', 'setores', 'turno_maq']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

bd_rot.rename(columns={
    'revezamento': 'REC_revezamento',
    'horas_extras': 'REC_horas_extras',
    'disp_mes': 'REC_disp_mes',
    'horas_indisp': 'REC_horas_indisp',
    'PER_TURNO': 'REC_PER_TURNO'
}, inplace=True)

# Carregando Calendário de Ferramentas
bd_calend_ferr = pd.read_excel(pasta_input / 'calend_ferramentas.xlsx', sheet_name='calend_ferramentas')
bd_calend_ferr['NEW_MOLDES'] = bd_calend_ferr['moldes'].str.upper() + "|" + bd_calend_ferr['unidade'].str.upper()

# Merge entre bd_rot e bd_calend_ferr
bd_rot = bd_rot.merge(
    bd_calend_ferr, 
    how='left', 
    left_on=['mes_ref', 'empresa', 'cod_ferramenta'], 
    right_on=['mes_ref', 'unidade', 'moldes']
)

colunas_excluir = ['moldes','unidade', 'setores']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

bd_rot.rename(columns={
    'horas_indisp': 'FER_horas_indisp',
    'disp_mes': 'FER_disp_mes',
    'NEW_MOLDES': 'moldes'
}, inplace=True)

# Preencher apenas os valores nulos de FER_horas_indisp com zero
bd_rot['FER_horas_indisp'] = bd_rot['FER_horas_indisp'].fillna(0)

# Calculando coluna HOR_REC_C1
# Se REC_revezamento for "SIM", calcula com TOT_HORAS_REV_CALEND, senão com TOT_HORAS_NOR_CALEND
# Calculando coluna HOR_REC_C1, se algum valor for nulo, retorna 0
bd_rot['HOR_REC_C1'] = np.where(
    bd_rot[['TOT_HORAS_REV_CALEND', 'REC_horas_extras', 'REC_horas_indisp', 'REC_PER_TURNO', 'REC_disp_mes', 'TOT_HORAS_NOR_CALEND']].isnull().any(axis=1),
    0,
    np.where(
        bd_rot['REC_revezamento'] == "SIM",
        ((bd_rot['TOT_HORAS_REV_CALEND'] + bd_rot['REC_horas_extras'] - bd_rot['REC_horas_indisp']) * bd_rot['REC_PER_TURNO']) * bd_rot['REC_disp_mes'],
        ((bd_rot['TOT_HORAS_NOR_CALEND'] + bd_rot['REC_horas_extras'] - bd_rot['REC_horas_indisp']) * bd_rot['REC_PER_TURNO']) * bd_rot['REC_disp_mes']
    )
).round(2)

# Calculando coluna HOR_FERR_C1
# Se REC_revezamento for "SIM", calcula com TOT_HORAS_REV_CALEND, senão com TOT_HORAS_NOR_CALEND
# Calculando coluna HOR_FER_C1, se algum valor for nulo, retorna 0, senão calcula conforme regra, com duas casas decimais
bd_rot['HOR_FER_C1'] = np.where(
    bd_rot[['TOT_HORAS_REV_CALEND', 'REC_horas_extras', 'FER_horas_indisp', 'TOT_HORAS_NOR_CALEND', 'FER_disp_mes']].isnull().any(axis=1),
    0,
    np.where(
        bd_rot['REC_revezamento'] == "SIM",
        (bd_rot['TOT_HORAS_REV_CALEND'] + (bd_rot['REC_horas_extras'] - bd_rot['FER_horas_indisp'])) * bd_rot['FER_disp_mes'],
        (bd_rot['TOT_HORAS_NOR_CALEND'] + (bd_rot['REC_horas_extras'] - bd_rot['FER_horas_indisp'])) * bd_rot['FER_disp_mes']
    )
).round(2)

# Criando coluna HOR_REC conforme a regra
# Se HOR_REC_C1 for null ou menor que 0, atribui 0; senão, atribui o valor de HOR_REC_C1
bd_rot['HOR_REC'] = np.where(
    bd_rot['HOR_REC_C1'].isnull() | (bd_rot['HOR_REC_C1'] < 0),
    0,
    bd_rot['HOR_REC_C1']
).round(2)

# Criando coluna HOR_FER conforme a regra
# Se HOR_FER_C1 for null ou menor que 0, atribui 0; senão, atribui o valor de HOR_FER_C1
bd_rot['HOR_FER'] = np.where(
    bd_rot['HOR_FER_C1'].isnull() | (bd_rot['HOR_FER_C1'] < 0),
    0,
    bd_rot['HOR_FER_C1']
).round(2)

# Criando coluna HOR_CAP conforme a regra
# Se HOR_REC for menor que HOR_FER, atribui HOR_REC; senão, atribui HOR_FER
bd_rot['HOR_CAP'] = np.where(
    bd_rot['HOR_REC'] < bd_rot['HOR_FER'],
    bd_rot['HOR_REC'],
    bd_rot['HOR_FER']
).round(2)

# Eliminando colunas para tornar mais clean a bd_rot
colunas_excluir = [
    'REC_revezamento', 'REC_horas_extras', 'REC_disp_mes', 'REC_horas_indisp',
    'TOT_HORAS_REV_CALEND', 'TOT_HORAS_NOR_CALEND', 'REC_PER_TURNO',
    'FER_horas_indisp', 'HOR_REC_C1', 'HOR_FER_C1', 'disp_mes'
]
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando a coluna prioridade do bd_rot para PRIORIDADE_ROTEIRO, para evitar confusão 
# com a prioridade da bd_mat_param
bd_rot.rename(columns={'prioridade': 'PRIORIDADE_ROTEIRO'}, inplace=True)

# Executando o merge entre bd_rot e bd_mat_param para trazer os campos unidade_fat e prioridade
bd_rot = bd_rot.merge(
    bd_mat_param, 
    how='left', 
    left_on=['empresa', 'cod_produto'], 
    right_on=['unidade_prod', 'produto']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = ['produto', 'unidade_prod', 'tipo_abast']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando colunas da matriz parametros que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'unidade_fat': 'MAT_PAR_unidade_fat',
    'prioridade': 'MAT_PAR_prioridade'
}, inplace=True)


# Executando o merge entre bd_rot e bd_prod_necs para trazer os campos de volumese e dados
bd_rot = bd_rot.merge(
    bd_prod_nec, 
    how='left', 
    left_on=['mes_ref', 'MAT_PAR_unidade_fat', 'cod_produto'], 
    right_on=['MES_REF', 'EMPRESA', 'COD_PROD']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = [
    'TOT_DIAS_NOR_CALEND','TOT_DIAS_REV_CALEND', 'moldes', 'MES_REF', 'EMPRESA',
    'UNID_PROD', 'COD_PROD', 'DESC_PROD', 'LINHA_PROD',
    'FAMILIA_PROD', 'TIPO_PROD', 'CURVA_ABC', 'CURVA_123'
]
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando da bd_roteiro que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'mes_ref': 'MES_REF',
    'empresa': 'EMPRESA',
    'cod_produto': 'COD_PROD',
    'descricao': 'DESC_PROD',
    'linha_prod': 'LINHA_PROD',
    'familia_prod': 'FAMILIA_PROD',
    'tipo_produto': 'TIPO_PROD',
    'COD_FERR': 'COD_FER_UNID',
    'cod_ferramenta': 'COD_FERR',
    'n_cavidade': 'N_CAVIDADE',
    'ciclo_molde': 'CICLO_MOLDE',
    'pcs_hr': 'PCS_HORA',
    'mo': 'MO',
    'alocacao_recurso': 'ALOC_REC',
    'grupo_setor': 'GRUPO_SETOR',
    'peso_produto_kg': 'PESO_PROD',
    'PRIORIDADE_ROTEIRO': 'PRIOR_ROT',
    'MAT_PAR_unidade_fat': 'UNID_FAT_MATPAR',
    'MAT_PAR_prioridade': 'PRIOR_MATPAR'
}, inplace=True)

# Executando o merge entre bd_rot e bd_matriz_regioes para trazer os campo região fat
bd_rot = bd_rot.merge(
    bd_matriz_regioes, 
    how='left', 
    left_on=['UNID_FAT_MATPAR'], 
    right_on=['Unidade']
)

# Elimnando campos que não são mais necessários da bd_rot
bd_rot = drop_colunas(bd_rot, ['Unidade'])

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'Região': 'REG_UNID_FAT'}, inplace=True)

# Executando o merge entre bd_rot e bd_matriz_regioes para trazer os campo região prod
bd_rot = bd_rot.merge(
    bd_matriz_regioes, 
    how='left', 
    left_on=['EMPRESA'], 
    right_on=['Unidade']
)

# Elimnando campos que não são mais necessários da bd_rot
bd_rot = drop_colunas(bd_rot, ['Unidade'])

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'Região': 'REG_UNID_PROD'}, inplace=True)

# Criando o campo MESMA_REG conforme a regra solicitada
bd_rot['MESMA_REG'] = np.where(
    bd_rot['REG_UNID_PROD'].isna(),
    "NAO",
    np.where(
        bd_rot['REG_UNID_FAT'] == bd_rot['REG_UNID_PROD'],
        "SIM",
        "NAO"
    )
)

# Substituir null (NaN) por 0 nos campos solicitados
campos_zerar = [
    'EST_SEG_PCS', 'EST_INI_PCS',
    'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'PREV_PCS', 'SALDO_PREV_PCS',
    'PESO_PROD_KG', 'EST_TRANS_PCS', 'SALDO_PREV_PROX_MES_PCS', 'PCS_NEC_PROD_MESMA_REG_SIM_NAO'
]
bd_rot[campos_zerar] = bd_rot[campos_zerar].fillna(0)

# Criando a coluna MES_REF_ANT reduzindo 1 mês de MES_REF
bd_rot['MES_REF_ANT'] = (bd_rot['MES_REF'] - pd.DateOffset(months=1)).dt.to_period('M').dt.to_timestamp()

# Renomeando da bd_roteiro que foram trazidas para a bd_rot
bd_rot.rename(columns={
    'EMPRESA': 'UNID_PROD',
    'UNID_FAT_MATPAR': 'UNID_FAT',
}, inplace=True)

# Importar aba matriz_regioes da planilha matriz limitantes
bd_matriz_limitantes = pd.read_excel(pasta_input / 'matriz_limitantes.xlsx', sheet_name='matriz_limitantes', dtype={'produto': str})

if 'mes_ref' in bd_matriz_limitantes.columns:
    bd_matriz_limitantes['mes_ref'] = pd.to_datetime(bd_matriz_limitantes['mes_ref'], errors='coerce', dayfirst=True)

# Executando o merge entre bd_rot e bd_matriz_limitantes para trazer os campo limitante_pcs
bd_rot = bd_rot.merge(
    bd_matriz_limitantes, 
    how='left',
    left_on=['MES_REF','UNID_FAT','COD_PROD'],
    right_on=['mes_ref','unidade','produto']
)

# Elimnando campos que não são mais necessários da bd_rot
colunas_excluir = ['mes_ref','unidade','produto']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Renomeando da campos que foram trazidos para a bd_rot
bd_rot.rename(columns={'limitante_pcs': 'LIMIT_PCS'}, inplace=True)

# Aplicando a regra no campo LIMIT_PCS conforme lógica Java
bd_rot['AVALIAR'] = bd_rot['PRIOR_MATPAR'].astype(str) + "|" + bd_rot['PRIOR_ROT'].astype(str)
bd_rot['LIMIT_PCS'] = np.where(
    (bd_rot['AVALIAR'] != "1|1") | (bd_rot['LIMIT_PCS'].isnull()),
    0,
    bd_rot['LIMIT_PCS']
)

# Formatando a coluna PRIOR_MATPAR para inteiro
bd_rot['PRIOR_MATPAR'] = bd_rot['PRIOR_MATPAR'].fillna(0).astype(int)

# Eliminar a coluna AVALIAR
colunas_excluir = ['AVALIAR', 'MES_REF_ANT']
bd_rot = drop_colunas(bd_rot, colunas_excluir)

# Criar ID_COMP
bd_rot['ID_COMP'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['COD_PROD'].str.upper()
)

# Criar ID_RECURSO
bd_rot['ID_RECURSO'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['ALOC_REC'].str.upper()
)

# Criar ID_FERRAMENTA
bd_rot['ID_FERRAMENTA'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_PROD'].str.upper() + "|" +
    bd_rot['COD_FER_UNID'].str.upper()
)

# Criar ID_PROD_UNID_FAT
bd_rot['ID_PROD_UNID_FAT'] = (
    bd_rot['MES_REF'].dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_FAT'].str.upper() + "|" +
    bd_rot['COD_PROD'].astype(str).str.upper()
)

# Criar ID_PROD_UNID_FAT_ANT já calculando o mês anterior no mesmo comando
bd_rot['ID_PROD_UNID_FAT_ANT'] = (
    (bd_rot['MES_REF'] - pd.DateOffset(months=1)).dt.strftime('%b%y').str.upper() + "|" +
    bd_rot['UNID_FAT'].str.upper()  + "|" +
    bd_rot['COD_PROD'].str.upper()
)
 
# Criar a coluna IND
# Classificação antiga
# bd_rot = bd_rot.sort_values(by=['MES_REF', 'UNID_FAT', 'COD_PROD', 'PRIOR_MATPAR', 'PRIOR_ROT', 'TIPO_PROD']).reset_index(drop=True)

# Classificação nova
bd_rot = bd_rot.sort_values(by=['MES_REF', 'TIPO_PROD', 'UNID_FAT', 'COD_PROD', 'PRIOR_MATPAR', 'PRIOR_ROT']).reset_index(drop=True)
bd_rot['IND'] = range(1, len(bd_rot) + 1)

# |||||||||||||||||||||||||||||||| ID_NUM_REC ||||||||||||||||||||||||||||||||
# Remover linhas onde ID_RECURSO é NaN
bd_rot = bd_rot[bd_rot['ID_RECURSO'].notna()].copy()

# Ordena primeiro por ID_RECURSO e depois por IND
# bd_rot = bd_rot.sort_values(by=['ID_RECURSO', 'IND']).reset_index(drop=True)

# Cria o índice incremental por ID_RECURSO, começando em 1
# bd_rot['ID_NUM_REC'] = (bd_rot.groupby('ID_RECURSO').cumcount() + 1).astype(int)

# Classificar crescente pela coluna IND
bd_rot = bd_rot.sort_values(by=['IND']).reset_index(drop=True)
# |||||||||||||||||||||||||||||||| ID_NUM_REC ||||||||||||||||||||||||||||||||

# |||||||||||||||||||||||||||||||| ID_NUM_FER ||||||||||||||||||||||||||||||||
# Remover linhas onde ID_FERRAMENTA é NaN
bd_rot = bd_rot[bd_rot['ID_FERRAMENTA'].notna()].copy()

# # Ordena primeiro por ID_NUM_FER e depois por IND
# bd_rot = bd_rot.sort_values(by=['ID_FERRAMENTA', 'IND']).reset_index(drop=True)

# # Cria o índice incremental por ID_NUM_FER, começando em 1
# bd_rot['ID_NUM_FER'] = (bd_rot.groupby('ID_FERRAMENTA').cumcount() + 1).astype(int)

# Classificar crescente pela coluna IND
bd_rot = bd_rot.sort_values(by=['IND']).reset_index(drop=True)
# |||||||||||||||||||||||||||||||| ID_NUM_FER ||||||||||||||||||||||||||||||||

# Criar coluna ID_CONC_REC = ID_RECURSO + "|" + ID_NUM_REC e ID_CONC_FER = ID_FERRAMENTA + "|" + ID_NUM_FER
# bd_rot['ID_CONC_REC'] = bd_rot['ID_RECURSO'] + "|" + bd_rot['ID_NUM_REC'].astype(str)
# bd_rot['ID_CONC_FER'] = bd_rot['ID_FERRAMENTA'] + "|" + bd_rot['ID_NUM_FER'].astype(str)

# Organizando Layout Final da para dados para bd_base_LTP
# Ordenando colunas
nova_ordem_colunas = [
    'MES_REF', 'UNID_FAT', 'UNID_PROD', 'MESMA_REG', 'PRIOR_MATPAR', 'PRIOR_ROT', 'COD_PROD', 'DESC_PROD', 'LINHA_PROD',
    'FAMILIA_PROD', 'TIPO_PROD', 'COD_FER_UNID', 'N_CAVIDADE', 'CICLO_MOLDE', 'PCS_HORA', 'MO', 'ALOC_REC', 
    'GRUPO_SETOR', 'PESO_PROD_KG', 'LOTE_ECON', 'QTD_EMB','LOTE_MIN', 'IND', 'ID_COMP', 'ID_RECURSO', 'ID_FERRAMENTA', 'ID_PROD_UNID_FAT', 'ID_PROD_UNID_FAT_ANT', 'HOR_REC', 'HOR_FER', 'HOR_CAP', 'PREV_PCS', 'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'SALDO_PREV_PCS', 'EST_TRANS_PCS', 'SALDO_PREV_PROX_MES_PCS', 'LIMIT_PCS'
    ]

# # Reordenando as colunas
bd_prod_rot = bd_rot[nova_ordem_colunas].reset_index(drop=True)

# Criar tabela com ID_PROD_UNID_FAT e MESMA_REG, filtrando PRIOR_MATPAR = 1 e PRIOR_ROT = 1
bd_prod_rot_PRIOR_11 = bd_prod_rot[
    (bd_prod_rot['PRIOR_MATPAR'] == 1) & (bd_prod_rot['PRIOR_ROT'] == 1)
][['ID_PROD_UNID_FAT', 'MESMA_REG']].drop_duplicates().reset_index(drop=True)
bd_prod_rot_PRIOR_11.rename(columns={'MESMA_REG': 'MESMA_REG_PRIOR_11'}, inplace=True)

# Fazer join entre bd_prod_rot e bd_prod_rot_PRIOR_11 para trazer MESMA_REG_PRIOR_11
bd_prod_rot = bd_prod_rot.merge(
    bd_prod_rot_PRIOR_11,
    how='left',
    on='ID_PROD_UNID_FAT'
)

# Substituir coluna MESMA_REG por MESMA_REG_PRIOR_11
bd_prod_rot['MESMA_REG'] = bd_prod_rot['MESMA_REG_PRIOR_11'].fillna(bd_prod_rot['MESMA_REG'])
# Eliminar coluna MESMA_REG_PRIOR_11
bd_prod_rot = drop_colunas(bd_prod_rot, ['MESMA_REG_PRIOR_11'])

# Criar bd_estrutura_filtrada, com base na bd_estruturas eliminando cod_insumo que não constam na coluna COD_PROD da bd_prod_rot
# codigos_validos = set(bd_prod_rot['COD_PROD'].unique())
# bd_estrutura_filtrada = bd_estruturas[bd_estruturas['cod_insumo'].isin(codigos_validos)].copy().reset_index(drop=True)

# Criar conjunto de tuplas válidas (COD_PROD, UNID_PROD)
codigos_validos = set(zip(bd_prod_rot['COD_PROD'], bd_prod_rot['UNID_PROD']))

# Filtrar estrutura com base em cod_insumo + empresa
bd_estrutura_filtrada = (
    bd_estruturas[
        bd_estruturas[['cod_insumo', 'empresa']].apply(tuple, axis=1).isin(codigos_validos)
    ]
    .copy()
    .reset_index(drop=True)
)

# Mesclar bd_base_LTP e bd_estrutura_filtrada por UNID_PROD, COD_PROD vs empresa e cod_prod_acabado
bd_prod_rot_estr = pd.merge(bd_prod_rot, bd_estrutura_filtrada, how='left', left_on=['UNID_PROD', 'COD_PROD'], right_on=['empresa', 'cod_prod_acabado']).reset_index(drop=True)

# Função para criar estrutura com fator estrutural
bd_estr_fator_estrutural = criar_estrutura_com_fator_estrutural(bd_estrutura_filtrada)

print("✅ Geração de tabelas de dados LTP concluída!")

✅ Geração de tabelas de dados LTP concluída!


In [5]:
########### TRAZER CAMPOS DA MATRIZ TRANSFERENCIA E ESTOQUE ORIGEM ###########
bd_LTP_NEC = bd_prod_rot.copy()

# Trazer para bd_produtos_nec as coluna de status da tabela bd_mat_transf_comp, colunas considera_estoq_origem e considera_estoq_triangulacao

 # Carregar matriz_transf_componentes
bd_mat_transf_comp = pd.read_excel(
    pasta_input / 'matriz_transf_componentes.xlsx',
    sheet_name='matriz_transf_componentes',
    dtype={'cod_produto': str}
)
bd_mat_transf_comp['cod_produto'] = bd_mat_transf_comp['cod_produto'].astype(str).str.strip().str.upper()
# Filtrar o DataFrame bd_mat_transf_comp para manter apenas as linhas onde 'validacao' == 'Sim'
bd_mat_transf_comp = bd_mat_transf_comp[bd_mat_transf_comp['validacao'].str.upper() == 'SIM']
bd_mat_transf_comp = bd_mat_transf_comp.reset_index(drop=True)

bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_mat_transf_comp[['unid_destino', 'unid_origem', 'unid_triangulacao', 'cod_produto', 'considera_estoq_origem', 'considera_estoq_triangulacao']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_destino', 'cod_produto']
)

# Trazer os valores de Estoque Inicial e Estoque Transferencia das unidades Origem
# Cria uma cópia da tabela para servir como origem
bd_produtos_origem = bd_LTP_NEC[['UNID_FAT', 'MES_REF', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS']].copy()

# Remover duplicatas bd_produtos_origem
bd_produtos_origem = bd_produtos_origem.drop_duplicates(subset=['UNID_FAT', 'MES_REF', 'COD_PROD']).reset_index(drop=True)

bd_produtos_origem = bd_produtos_origem.rename(columns={
    'UNID_FAT': 'UNID_FAT_ORIGEM',
    'MES_REF': 'MES_REF_ORIGEM',
    'COD_PROD': 'COD_PROD_ORIGEM',
    'EST_INI_PCS': 'EST_INI_PCS_ORIGEM',
    'EST_TRANS_PCS': 'EST_TRANS_PCS_ORIGEM'
})

# Eliminar duplicatas da tabela de origem
bd_produtos_origem = bd_produtos_origem.drop_duplicates(subset=['UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM', 'EST_INI_PCS_ORIGEM', 'EST_TRANS_PCS_ORIGEM'])

# Faz o merge na própria tabela, buscando os valores da origem
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_produtos_origem,
    how='left',
    left_on=['unid_origem', 'MES_REF', 'COD_PROD'],
    right_on=['UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM']
)

# Criar uma tabela com cópia na bd_produtos_nec ter somente as colunas de interesse e buscar os valores de estoque origem e estoque triangulação
bd_produtos_estoque_origem_triangulacao = bd_LTP_NEC
colunas_excluir = ['considera_estoq_origem', 'considera_estoq_triangulacao']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)


# Trazer para bd_produtos_nec as colunas de status da tabela bd_mat_transf_comp, colunas considera_estoq_origem e considera_estoq_triangulacao
bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.merge(
    bd_mat_transf_comp[['unid_origem', 'cod_produto', 'considera_estoq_origem']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_origem', 'cod_produto']
)

# Excluir as colunas unid_triangulacao e cod_produto
colunas_excluir = ['unid_triangulacao', 'cod_produto']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)

bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.merge(
    bd_mat_transf_comp[['unid_triangulacao', 'cod_produto', 'considera_estoq_triangulacao']],
    how='left',
    left_on=['UNID_FAT', 'COD_PROD'],
    right_on=['unid_triangulacao', 'cod_produto']
)

# Excluir as colunas unid_destino e cod_produto
colunas_excluir = ['unid_triangulacao', 'cod_produto']
bd_produtos_estoque_origem_triangulacao = drop_colunas(bd_produtos_estoque_origem_triangulacao, colunas_excluir)

bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao[['MES_REF', 'UNID_FAT', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS','considera_estoq_origem', 'considera_estoq_triangulacao']]

# Eliminar duplicatas da bd_produtos_estoque_origem_triangulacao
bd_produtos_estoque_origem_triangulacao = bd_produtos_estoque_origem_triangulacao.drop_duplicates(subset=['MES_REF', 'UNID_FAT', 'COD_PROD']).reset_index(drop=True, inplace=True)

# Trazer os valores de Estoque Inicial e Estoque Transferencia das unidades Triangulação
# Cria uma cópia da tabela para servir como Triangulação
bd_produtos_triangulacao = bd_LTP_NEC[['UNID_FAT', 'MES_REF', 'COD_PROD', 'EST_INI_PCS', 'EST_TRANS_PCS']].copy()

# Renomeando e organizando colunas bd_produtos_triangulacao
bd_produtos_triangulacao = bd_produtos_triangulacao.rename(columns={
    'UNID_FAT': 'UNID_FAT_TRIANG',
    'MES_REF': 'MES_REF_TRIANG',
    'COD_PROD': 'COD_PROD_TRIANG',
    'EST_INI_PCS': 'EST_INI_PCS_TRIANG',
    'EST_TRANS_PCS': 'EST_TRANS_PCS_TRIANG'
})

# Remover Duplicatas da tabela de triangulação
bd_produtos_triangulacao = bd_produtos_triangulacao.drop_duplicates(subset=['UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG']).reset_index(drop=True)

# Faz o merge na própria tabela, buscando os valores da TRIANG
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_produtos_triangulacao,
    how='left',
    left_on=['unid_triangulacao', 'MES_REF', 'COD_PROD'],
    right_on=['UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG']
)

# Somar valores Estoque Inicial e Transferencia da Origem
def ORI_TOT_PCS(row):
    if str(row.get('considera_estoq_origem', '')).strip().upper() == 'SIM':
        return row.get('EST_INI_PCS_ORIGEM', 0) + row.get('EST_TRANS_PCS_ORIGEM', 0)
    else:
        return 0
bd_LTP_NEC['ORI_TOT_PCS'] = bd_LTP_NEC.apply(ORI_TOT_PCS, axis=1)

# Somar valores Estoque Inicial e Transferencia da Triangulação
def TRIANG_TOT_PCS(row):
    if str(row.get('considera_estoq_triangulacao', '')).strip().upper() == 'SIM':
        return row.get('EST_INI_PCS_TRIANG', 0) + row.get('EST_TRANS_PCS_TRIANG', 0)
    else:
        return 0
bd_LTP_NEC['TRIANG_TOT_PCS'] = bd_LTP_NEC.apply(TRIANG_TOT_PCS, axis=1)

# Eliminar colunas desnecessárias da bd_LTP_NEC
colunas_excluir = [
    'unid_destino', 'unid_origem', 'unid_triangulacao', 'cod_produto',
    'considera_estoq_origem', 'considera_estoq_triangulacao',
    'UNID_FAT_ORIGEM', 'MES_REF_ORIGEM', 'COD_PROD_ORIGEM',
    'EST_INI_PCS_ORIGEM', 'EST_TRANS_PCS_ORIGEM',
    'UNID_FAT_TRIANG', 'MES_REF_TRIANG', 'COD_PROD_TRIANG',
    'EST_INI_PCS_TRIANG', 'EST_TRANS_PCS_TRIANG'
]
bd_LTP_NEC = drop_colunas(bd_LTP_NEC, colunas_excluir)

# Aplicar zero nos campos vazios de NEC_PCS, ORI_TOT_PCS e TRIANG_TOT_PCS
colunas_preencher = ['ORI_TOT_PCS', 'TRIANG_TOT_PCS']
bd_LTP_NEC[colunas_preencher] = bd_LTP_NEC[colunas_preencher].fillna(0).astype(float)

# Filtrar bd_LTP_NEC pelo campo MAT_PAR_prioridade > 0
bd_LTP_NEC = bd_LTP_NEC[bd_LTP_NEC['PRIOR_MATPAR'] > 0].reset_index(drop=True)

#*****************************************# ID_ULT_PRIORI #*****************************************
# Criar a tabela bd_Ultimo_Roteiro_MAT_PAR com os campos ID_PROD_UNID_FAT, PRIOR_MATPAR e PRIOR_ROT, eliminando as duplicatas
bd_Ultimo_Roteiro_MAT_PAR = bd_LTP_NEC[['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT']]
bd_Ultimo_Roteiro_MAT_PAR = bd_Ultimo_Roteiro_MAT_PAR.sort_values(
    by=['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'],
    ascending=[True, True, False]
).reset_index(drop=True)
bd_Ultimo_Roteiro_MAT_PAR = bd_Ultimo_Roteiro_MAT_PAR.drop_duplicates(subset='ID_PROD_UNID_FAT', keep='first').reset_index(drop=True)

# Criar a coluna ID_ULT_PRIORI fazendo o merge entre bd_LTP_NEC e bd_Ultimo_Roteiro_MAT_PAR para trazer dados a coluna ID_PROD_UNID_FAT
bd_LTP_NEC = bd_LTP_NEC.merge(
    bd_Ultimo_Roteiro_MAT_PAR[['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT']].rename(columns={'ID_PROD_UNID_FAT': 'ID_ULT_PRIORI'}),
    how='left',
    left_on=['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'],
    right_on=['ID_ULT_PRIORI', 'PRIOR_MATPAR', 'PRIOR_ROT']
)

bd_LTP_NEC = bd_LTP_NEC.sort_values(['IND']).reset_index(drop=True)

# Zerar valores campo SALDO_PREV_PROX_MES_PCS, considerando se MESMA_REG for igual a "SIM", manter valor, se MESMA_REG for "NAO", zerar o valor
bd_LTP_NEC['SALDO_PREV_PROX_MES_PCS'] = np.where(
    bd_LTP_NEC['MESMA_REG'] == 'NAO',
    bd_LTP_NEC['SALDO_PREV_PROX_MES_PCS'],
    0
)

# Criando cópia dos campos e adicinar nos rótulos, no incio do nome LTP
campos_copiar = [
    'EST_SEG_PCS', 'EST_INI_PCS', 'CART_ARR_MES_ANT', 'CART_MES_ATUAL', 'SALDO_PREV_PCS', 'EST_TRANS_PCS','SALDO_PREV_PROX_MES_PCS'
]

for col in campos_copiar:
    bd_LTP_NEC['LTP_' + col] = bd_LTP_NEC[col]
    
# Adicionar coluna LTP_COMP_NEC_PCS com valor 0
bd_LTP_NEC['LTP_COMP_NEC_PCS'] = 0

# --- Utilizando Flags Parametros Lote Minimo e Multiplo Embalagens e Calculando os campos VAR_NEC1, VAR_NEC2 e VAR_NEC3 ---
def carregar_flags_ltp(pasta_painel):
    def carregar_planilha_segura(caminho: Path, **kwargs):
        try:
            return load_workbook(caminho, **kwargs)
        except PermissionError:
            with tempfile.NamedTemporaryFile(suffix=caminho.suffix, delete=False) as tmp:
                shutil.copy2(caminho, tmp.name)
                return load_workbook(tmp.name, **kwargs)

    def obter_valor_nome_definido(wb, nome_definido):
        nome_planilha, intervalo = next(nome_definido.destinations)
        planilha = wb[nome_planilha]
        coluna_min, linha_min, _, _ = range_boundaries(intervalo)
        return planilha.cell(linha_min, coluna_min).value

    arquivo_painel = pasta_painel / 'Painel_LTP.xlsm'
    planilha_ltp = carregar_planilha_segura(arquivo_painel, data_only=True)
    nome_lote_min = planilha_ltp.defined_names['FlagLoteMinimo']
    nome_multiplo_emb = planilha_ltp.defined_names['FlagMultiploEmb']
    lote_min = obter_valor_nome_definido(planilha_ltp, nome_lote_min)
    multiplo_emb = obter_valor_nome_definido(planilha_ltp, nome_multiplo_emb)
    return str(lote_min).strip().upper(), str(multiplo_emb).strip().upper()

lote_min_flag, multiplo_emb_flag = carregar_flags_ltp(pasta_painel)

# Mover Colunas ID_RECURSO e id_FERRAMENTA para o final do DataFrame, depois da coluna LTP_COMP_NEC_PCS
bd_LTP_NEC = bd_LTP_NEC[
    [col for col in bd_LTP_NEC.columns if col not in ['ID_RECURSO', 'ID_FERRAMENTA']] +
    ['ID_RECURSO', 'ID_FERRAMENTA']
]

# Mover colunas HOR_REC, HOR_FER, HOR_CAP para o final do DataFrame, depois da coluna LTP_COMP_NEC_PCS
bd_LTP_NEC = bd_LTP_NEC[
    [col for col in bd_LTP_NEC.columns if col not in ['HOR_REC', 'HOR_FER']] +
    ['HOR_REC', 'HOR_FER']
]

# Excluir coluna HOR_CAP
bd_LTP_NEC = drop_colunas(bd_LTP_NEC, ['HOR_CAP'])

# criar coluna COMP com valor 'SIM' ou 'NAO' dependendo se o COD_PROD está na tabela bd_estruturas
insumos = set(bd_estruturas['cod_insumo'].astype(str).str.strip().str.upper())
bd_LTP_NEC['COMP'] = (bd_LTP_NEC['COD_PROD'].astype(str).str.strip().str.upper().isin(insumos).map({True: 'SIM', False: 'NAO'}))

print("✅ Primeiro Calculo NEC_PCS e Distribuição Capacidade concluídos!")

✅ Primeiro Calculo NEC_PCS e Distribuição Capacidade concluídos!


In [6]:
VALIDOS = {"RECURSO", "FERRAMENTA"}

def carregar_cortes(pasta_painel: Path):

    def carregar_planilha_segura(caminho: Path, **kwargs):
        try:
            return load_workbook(caminho, **kwargs)
        except PermissionError:
            with tempfile.NamedTemporaryFile(suffix=caminho.suffix, delete=False) as tmp:
                shutil.copy2(caminho, tmp.name)
                return load_workbook(tmp.name, **kwargs)

    def obter_valor_nome_definido(wb, nome_definido):
        nome_planilha, intervalo = next(nome_definido.destinations)
        planilha = wb[nome_planilha]
        coluna_min, linha_min, _, _ = range_boundaries(intervalo)
        return planilha.cell(linha_min, coluna_min).value

    def limpar_valor(x):
        if x is None:
            return ""
        if isinstance(x, float) and math.isnan(x):
            return ""
        return str(x).strip().upper()

    arquivo_painel = pasta_painel / 'Painel_LTP.xlsm'
    planilha_ltp = carregar_planilha_segura(arquivo_painel, data_only=True)

    try:
        nome_primeiro_corte = planilha_ltp.defined_names['rgPrimeiroCorte']
        nome_segundo_corte  = planilha_ltp.defined_names['rgSegundoCorte']
    except KeyError as e:
        raise RuntimeError(
            "Ranges nomeadas 'rgPrimeiroCorte' ou 'rgSegundoCorte' não encontradas no Painel_LTP.xlsm."
        ) from e

    rgPrimeiroCorte = limpar_valor(obter_valor_nome_definido(planilha_ltp, nome_primeiro_corte))
    rgSegundoCorte  = limpar_valor(obter_valor_nome_definido(planilha_ltp, nome_segundo_corte))

    # 🔴 VALIDAÇÃO OBRIGATÓRIA
    if not rgPrimeiroCorte or not rgSegundoCorte:
        raise ValueError(
            "Defina obrigatoriamente os campos 'Primeiro Corte' e 'Segundo Corte' no Painel_LTP.\n"
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgPrimeiroCorte not in VALIDOS:
        raise ValueError(
            f"Valor inválido em rgPrimeiroCorte: '{rgPrimeiroCorte}'. "
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgSegundoCorte not in VALIDOS:
        raise ValueError(
            f"Valor inválido em rgSegundoCorte: '{rgSegundoCorte}'. "
            "Valores permitidos: RECURSO ou FERRAMENTA."
        )

    if rgPrimeiroCorte == rgSegundoCorte:
        raise ValueError(
            "Primeiro Corte e Segundo Corte não podem ser iguais.\n"
            "Escolha uma sequência válida entre RECURSO e FERRAMENTA."
        )

    return rgPrimeiroCorte, rgSegundoCorte

# chamada
rgPrimeiroCorte, rgSegundoCorte = carregar_cortes(pasta_painel)

In [7]:
# Se IND repetido, parar o código e gerar print de aviso, indicando quais são os IND repetidos para análise
if bd_LTP_NEC['IND'].duplicated().any():
    inds_duplicados = bd_LTP_NEC[bd_LTP_NEC['IND'].duplicated()]['IND'].unique()
    print(f"⚠️ Atenção: IND duplicados encontrados! IND's repetidos: {inds_duplicados}")
    raise ValueError("IND duplicados encontrados. Verifique os dados antes de continuar.")

In [8]:

# LOG: ======================================================================================
# LOG: Definições do Log de Auditoria de Cortes
# LOG: ======================================================================================

COLS_ECO_ANTES = ["MES_REF","UNID_FAT","UNID_PROD","MESMA_REG","COD_PROD","DESC_PROD","TIPO_PROD","COD_FER_UNID","PCS_HORA","ALOC_REC","LOTE_ECON","QTD_EMB","LOTE_MIN","IND","LTP_EST_SEG_PCS","LTP_EST_INI_PCS","LTP_CART_ARR_MES_ANT","LTP_CART_MES_ATUAL","LTP_SALDO_PREV_PCS","LTP_EST_TRANS_PCS","LTP_SALDO_PREV_PROX_MES_PCS","LTP_COMP_NEC_PCS","NEC_PCS","NEC_HR","NEC_ATEND_PCS","NEC_ATEND_HR","NEC_NAO_ATEND_PCS","NEC_NAO_ATEND_HR","NEC_ESTOURO_PCS","NEC_ESTOURO_HR","NEC_N_ATEND_PCS_REC","NEC_N_ATEND_PCS_FER","NEC_ESTOURO_PCS_REC","NEC_ESTOURO_HR_REC","NEC_ESTOURO_PCS_FER","NEC_ESTOURO_HR_FER","ET_PCS","C_ARR_PCS","C_AT_PCS","PV_PCS","PV_PROX_PCS","ES_PCS","DIF_LM_PCS","DIF_EMB_PCS","C_ARR_HR","C_AT_HR","PV_HR","PV_PROX_HR","ES_HR","DIF_LM_HR","DIF_EMB_HR","RASTREABILIDADE","PCS_HORA_PI","COD_INSUMO","FATOR_ESTRUTURAL","NEC_ATEND_PCS_PI","NEC_ATEND_HR_PI","MODO_CORTE","RATEIO","CORTE_HOR","CORTE_PCS","SALDO_PCS","LTP_SALDO_PREV_PCS_ANTES"]

COLS_ECO_DEPOIS = ["LTP_EST_SEG_PCS","LTP_EST_INI_PCS","LTP_CART_ARR_MES_ANT","LTP_CART_MES_ATUAL","LTP_SALDO_PREV_PCS","LTP_EST_TRANS_PCS","LTP_SALDO_PREV_PROX_MES_PCS","LTP_COMP_NEC_PCS","NEC_PCS","NEC_HR","NEC_ATEND_PCS","NEC_ATEND_HR","NEC_NAO_ATEND_PCS","NEC_NAO_ATEND_HR","NEC_ESTOURO_PCS","NEC_ESTOURO_HR","NEC_N_ATEND_PCS_REC","NEC_N_ATEND_PCS_FER","NEC_ESTOURO_PCS_REC","NEC_ESTOURO_HR_REC","NEC_ESTOURO_PCS_FER","NEC_ESTOURO_HR_FER","ET_PCS","C_ARR_PCS","C_AT_PCS","PV_PCS","PV_PROX_PCS","ES_PCS","DIF_LM_PCS","DIF_EMB_PCS","C_ARR_HR","C_AT_HR","PV_HR","PV_PROX_HR","ES_HR","DIF_LM_HR","DIF_EMB_HR","RASTREABILIDADE","PCS_HORA_PI","COD_INSUMO","FATOR_ESTRUTURAL","NEC_ATEND_PCS_PI","NEC_ATEND_HR_PI","MODO_CORTE","RATEIO","CORTE_HOR","CORTE_PCS","SALDO_PCS","LTP_SALDO_PREV_PCS_ANTES","LTP_SALDO_PREV_PCS_DEPOIS"]

DEFAULTS_ECO_LOG = {
    "RATEIO": 0.0,
    "CORTE_HOR": 0.0,
    "CORTE_PCS": 0.0,
    "SALDO_PCS": np.nan,
    "LTP_SALDO_PREV_PCS_ANTES": np.nan,
    "LTP_SALDO_PREV_PCS_DEPOIS": np.nan,
    "PCS_HORA_PI": np.nan,
    "COD_INSUMO": np.nan,
    "FATOR_ESTRUTURAL": np.nan,
    "NEC_ATEND_PCS_PI": np.nan,
    "NEC_ATEND_HR_PI": np.nan,
    "RASTREABILIDADE": np.nan,
    "MODO_CORTE": np.nan,
}

MAP_HR_TO_LTP = {
    "C_ARR_HR": "LTP_CART_ARR_MES_ANT",
    "C_AT_HR": "LTP_CART_MES_ATUAL",
    "PV_HR": "LTP_SALDO_PREV_PCS",
    "PV_PROX_HR": "LTP_SALDO_PREV_PROX_MES_PCS",
    "ES_HR": "LTP_EST_SEG_PCS",
}

def garantir_colunas_log(df, cols):
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            df[col] = DEFAULTS_ECO_LOG.get(col, np.nan)
    return df

def montar_snapshot_antes(df):
    df = garantir_colunas_log(df, COLS_ECO_ANTES)
    snap = df[COLS_ECO_ANTES].copy()
    snap = snap.rename(columns={col: f"ANT_{col}" for col in COLS_ECO_ANTES})
    return snap

def montar_snapshot_depois(df):
    df = garantir_colunas_log(df, ["IND"] + COLS_ECO_DEPOIS)
    snap = df[["IND"] + COLS_ECO_DEPOIS].copy()
    snap = snap.rename(columns={col: f"DEP_{col}" for col in COLS_ECO_DEPOIS})
    return snap

def registrar_evento_log(logs_buffer, bd_eco_antes, bd_eco_depois, metadata_evento):
    ant = montar_snapshot_antes(bd_eco_antes)
    dep = montar_snapshot_depois(bd_eco_depois)

    ant["LOG_IND"] = ant["ANT_IND"]
    dep["LOG_IND"] = dep["IND"]
    dep = dep.drop(columns=["IND"])

    log_evento = ant.merge(dep, how="left", on="LOG_IND")

    for k, v in metadata_evento.items():
        log_evento[k] = v

    logs_buffer.append(log_evento)

# LOG: Acumulador do log
logs_cortes = []
id_evento_corte = 0

# ============================ 0) Pré-processo: Mês ============================

# Normalizar Data
bd_LTP_NEC["MES_REF"] = bd_LTP_NEC["MES_REF"].dt.normalize()

# Criar uma lista de MES_REF únicos (ordenar)
mes_refs = sorted(bd_LTP_NEC["MES_REF"].unique().tolist())

# Loop sobre os meses
for mes_ref in mes_refs:

    if mes_ref != mes_refs[0]:
        continue

    # ======================================================================================
    # 1) Montar base mensal (bd_LTP_MES) e ativar IND como índice técnico
    # ======================================================================================
    bd_LTP_MES = bd_LTP_NEC.loc[bd_LTP_NEC["MES_REF"] == mes_ref].reset_index(drop=True)

    # ======================================================================================
    # 2) Recalcular modelo completo
    # ======================================================================================
    bd_LTP_MES = calc_nec_pcs_hr(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
    bd_LTP_MES, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
    bd_LTP_MES = calcular_demais_campos(bd_LTP_MES)

    # *************************# Explosão da Estrutura e necessidade de componentes #**************************
    bd_ltp_estrutura_explodida = explodir_estrutura_ltp(bd_estrutura_filtrada, bd_LTP_MES)
    bd_nec_comp_expl = calcular_explosao_necessidades(bd_ltp_estrutura_explodida, bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
    bd_LTP_MES = atualizar_ltp_comp_nec_pcs(bd_LTP_MES, bd_nec_comp_expl)

    # *********************# Recalculando NEC_PCS e demais campos, após explosão componentes #**************************
    bd_LTP_MES = calc_nec_pcs_hr(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
    bd_LTP_MES, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
    bd_LTP_MES = calcular_demais_campos(bd_LTP_MES)

    # FIXME - Gerar um Snapshot do bd_LTP_MES antes dos cortes, para análise de impacto pós-corte (comparar NEC_PCS, NEC_HR, etc.)
    bd_LTP_MES_snapshot = bd_LTP_MES.copy(deep=True)

    # ======================================================================================
    # 3) Preparar base de cortes e decomposição (igual seu fluxo)
    # ======================================================================================

    # Resolver dimensão dinamicamente conforme painel
    if rgPrimeiroCorte == "RECURSO":
        col_rec_fer = "ALOC_REC"
        col_rec_fer_estouro_pcs = "NEC_ESTOURO_PCS_REC"
        fn_cria_base_cortes = cria_bd_mat_cortes_REC
        # LOG: Contexto da dimensão ativa para auditoria
        dimensao_corte = "REC"
    else:
        col_rec_fer = "COD_FER_UNID"
        col_rec_fer_estouro_pcs = "NEC_ESTOURO_PCS_FER"
        fn_cria_base_cortes = cria_bd_mat_cortes_FER
        # LOG: Contexto da dimensão ativa para auditoria
        dimensao_corte = "FER"

    bd_MAT_HOR_CORTES = fn_cria_base_cortes(bd_LTP_MES)

    # FIXME - Gerar um Snapshot da base de cortes antes do início dos cortes
    bd_MAT_HOR_CORTES_snapshot = bd_MAT_HOR_CORTES.copy(deep=True)

    # Manter apenas as linhas da coluna CORTE_HR maiores que 100% na bd_MAT_HOR_CORTES, para otimizar o processo de cortes (foco nos casos mais críticos)
    bd_MAT_HOR_CORTES = bd_MAT_HOR_CORTES.loc[bd_MAT_HOR_CORTES["CORTE_HR"] > 100].reset_index(drop=True)

    bd_LTP_MES = calcular_decomposicao_nec_pcs_para_precisao_corte(bd_LTP_MES)

    # ======================================================================================
    # 4) Driver de cortes (dinâmico para RECURSO ou FERRAMENTA)
    # ======================================================================================

    contador = defaultdict(int)
    LIMITE = 5

    while True:

        # Recriar base de cortes atualizada
        bd_MAT_HOR_CORTES = fn_cria_base_cortes(bd_LTP_MES)

        # Filtrar apenas > 100%
        bd_MAT_HOR_CORTES = bd_MAT_HOR_CORTES.loc[
            bd_MAT_HOR_CORTES["CORTE_HR"] > 100
        ].reset_index(drop=True)

        # Condição de parada
        if bd_MAT_HOR_CORTES.empty:
            break

        # Ordena da mais crítica para a menor
        bd_MAT_HOR_CORTES = bd_MAT_HOR_CORTES.sort_values(
            by="CORTE_HR", ascending=False
        ).reset_index(drop=True)

        # Remove chaves que já rodaram 5 vezes
        bd_MAT_HOR_CORTES = bd_MAT_HOR_CORTES.loc[
            bd_MAT_HOR_CORTES.apply(
                lambda x: contador[(x[col_rec_fer], x["UNID_PROD"])] < LIMITE,
                axis=1
            )
        ].reset_index(drop=True)

        # Se não sobrou nenhuma chave válida → encerra o while
        if bd_MAT_HOR_CORTES.empty:
            break

        # Agora pega a próxima chave válida
        chave_dim = bd_MAT_HOR_CORTES.at[0, col_rec_fer]
        unid_prod = bd_MAT_HOR_CORTES.at[0, "UNID_PROD"]

        # Incrementa contador da chave
        chave = (chave_dim, unid_prod)
        contador[chave] += 1
        # LOG: Iteração da chave atual
        iteracao_chave = contador[chave]

        # Recorte do LTP do mês pela dimensão ativa (RECURSO ou FERRAMENTA) combinado com UNID_PROD, para montar o pacote da chave atual
        filtro_dim = (
            (bd_LTP_MES[col_rec_fer] == chave_dim) &
            (bd_LTP_MES["UNID_PROD"] == unid_prod)
        )

        # Gerar o recorte do LTP para a dimensão e produto atuais
        bd_LTP_MES_DIM = bd_LTP_MES.loc[filtro_dim].copy()

        # Remove apenas PI sem necessidade nem estouro (NEC_ATEND_PCS + estouro = 0)
        bd_LTP_MES_DIM = bd_LTP_MES_DIM[
            (bd_LTP_MES_DIM["TIPO_PROD"] != "PI") |
            (
                (bd_LTP_MES_DIM["NEC_ATEND_PCS"] +
                bd_LTP_MES_DIM[col_rec_fer_estouro_pcs]) > 0
            )
        ]

        # Se o recorte do LTP para a dimensão e produto atuais estiver vazio, pular para próxima iteração (próxima chave)
        if bd_LTP_MES_DIM.empty:
            continue

        # Somar NEC_ATEND_PCS com TIPO_PROD = 'PI' para decidir modo PI ou PA
        soma_nec_atend_pcs_pi = bd_LTP_MES_DIM.loc[
            bd_LTP_MES_DIM["TIPO_PROD"] == "PI", "NEC_ATEND_PCS"
        ].sum()

        # Somar NEC_ESTOURO_PCS conforme painel para decidir modo PI ou PA
        soma_nec_estouro_pi = bd_LTP_MES_DIM.loc[
            bd_LTP_MES_DIM["TIPO_PROD"] == "PI", col_rec_fer_estouro_pcs
        ].sum()

        # Variavel validando processo PI
        valor_pi = soma_nec_atend_pcs_pi + soma_nec_estouro_pi

        # Filtrar a base de cortes para essa chave e somar horas totais de corte
        bd_MAT_HOR_CORTES_DIM = bd_MAT_HOR_CORTES[
            (bd_MAT_HOR_CORTES["UNID_PROD"] == unid_prod) &
            (bd_MAT_HOR_CORTES[col_rec_fer] == chave_dim)
        ]

        val_hor_objetivo_mat_corte = float(bd_MAT_HOR_CORTES_DIM["CORTE_HR"].sum())

        if val_hor_objetivo_mat_corte <= 0:
            continue

        # ======================================================================================
        # 5) Montagem do ecossistema (PI ou PA)
        # ======================================================================================

        # ----------------------------- MODO PI -----------------------------
        if ("PI" in bd_LTP_MES_DIM["TIPO_PROD"].values) and (valor_pi > 0):

            # Listar os COD_PROD e UNID_PROD que sejam PI, eliminando duplicatas e NEC_PCS = 0
            pi_itens = (
                bd_LTP_MES_DIM.loc[
                    bd_LTP_MES_DIM["TIPO_PROD"] == "PI",
                    ["COD_PROD", "UNID_PROD", "PCS_HORA", "NEC_PCS"]
                ]
                .drop_duplicates()
                .reset_index(drop=True)
            )
            pi_itens = pi_itens.loc[pi_itens["NEC_PCS"] > 0].reset_index(drop=True)

            pi_itens["RASTREABILIDADE"] = pi_itens["COD_PROD"] + "|" + pi_itens["UNID_PROD"]
            pi_itens["COD_INSUMO"] = pi_itens["COD_PROD"]

            # Ecossistema PI começa com o recorte da dimensão
            bd_ECO_PI = bd_LTP_MES_DIM.copy()

            for _, pi in pi_itens.iterrows():

                cod_pi = pi["COD_PROD"]
                unid_pi = pi["UNID_PROD"]
                rastreabilidade = pi["RASTREABILIDADE"]
                pcs_hora_pi = pi["PCS_HORA"]
                cod_insumo = pi["COD_INSUMO"]

                # PAs originados por esse PI na estrutura
                lista_itens_pa_estr_expl = bd_ltp_estrutura_explodida[
                    (bd_ltp_estrutura_explodida["COD_INSUMO"] == cod_pi) &
                    (bd_ltp_estrutura_explodida["UNID_PROD"] == unid_pi)
                ][["COD_PROD", "UNID_PROD"]].drop_duplicates()

                for _, pa in lista_itens_pa_estr_expl.iterrows():
                    cod_pa = pa["COD_PROD"]
                    unid_prod_estrutura = pa["UNID_PROD"]

                    filtro_estrutura = (
                        (bd_LTP_MES["COD_PROD"] == cod_pa) &
                        (bd_LTP_MES["UNID_PROD"] == unid_prod_estrutura)
                    )

                    linhas_filtradas_pa_LTP = bd_LTP_MES.loc[filtro_estrutura].copy()
                    if linhas_filtradas_pa_LTP.empty:
                        continue

                    linhas_filtradas_pa_LTP["RASTREABILIDADE"] = rastreabilidade
                    linhas_filtradas_pa_LTP["PCS_HORA_PI"] = pcs_hora_pi
                    linhas_filtradas_pa_LTP["COD_INSUMO"] = cod_insumo

                    bd_ECO_PI = pd.concat([bd_ECO_PI, linhas_filtradas_pa_LTP], ignore_index=False)
            

            # ---------------------------------------------------------------------------------------------------------------
            # FIXME - Validar e descobrir porque tem casos que o COD_INSUMO não está vindo na base de estrutura explodida, mesmo existindo o COD_PROD como insumo. Pode ser algo relacionado a UNID_PROD, ou algum detalhe da estrutura. Importante validar para garantir que o FATOR_ESTRUTURAL seja trazido corretamente e o corte seja preciso.
            print("DEBUG PI:", f"chave_dim={chave_dim}", f"unid_prod={unid_prod}")
            print("tem COD_INSUMO no final? ", "COD_INSUMO" in bd_ECO_PI.columns)
            print("shape bd_ECO_PI =", bd_ECO_PI.shape)

            if "COD_INSUMO" not in bd_ECO_PI.columns:
                raise Exception(f"ERRO | coluna COD_INSUMO ausente | chave_dim={chave_dim} | unid_prod={unid_prod}")

            qtd_cod_insumo_preenchido = bd_ECO_PI["COD_INSUMO"].notna().sum()
            print("qtd linhas com COD_INSUMO preenchido =", qtd_cod_insumo_preenchido)

            if qtd_cod_insumo_preenchido == 0:
                raise Exception(f"ERRO | COD_INSUMO sem preenchimento | chave_dim={chave_dim} | unid_prod={unid_prod}")
            # ---------------------------------------------------------------------------------------------------------------

            # Trazer para ecossistema o FATOR_ESTRUTURAL (COD_PROD, UNID_PROD, COD_INSUMO)
            bd_ECO_PI = bd_ECO_PI.merge(
                bd_estr_fator_estrutural[["COD_PROD", "UNID_PROD", "COD_INSUMO", "FATOR_ESTRUTURAL"]],
                how="left",
                on=["COD_PROD", "UNID_PROD", "COD_INSUMO"]
            )

            # NEC_ATEND_PCS_PI = NEC_ATEND_PCS * FATOR_ESTRUTURAL
            bd_ECO_PI["NEC_ATEND_PCS_PI"] = bd_ECO_PI["NEC_ATEND_PCS"] * bd_ECO_PI["FATOR_ESTRUTURAL"]

            # NEC_ATEND_HR_PI = NEC_ATEND_PCS_PI / PCS_HORA_PI
            bd_ECO_PI["NEC_ATEND_HR_PI"] = bd_ECO_PI["NEC_ATEND_PCS_PI"] / bd_ECO_PI["PCS_HORA_PI"]

            # Definir colunas-alvo para o motor de corte
            bd_ECO = bd_ECO_PI
            col_hora_alvo = "NEC_ATEND_HR_PI"

            # Se quiser enxergar explicitamente o modo aplicado no pacote (debug)
            bd_ECO["MODO_CORTE"] = "PI"

            # Eliminar Duplicatas por IND, mantendo a primeira ocorrência (prioriza manter o PI e os PAs relacionados, caso haja duplicatas)
            bd_ECO = bd_ECO.drop_duplicates(subset='IND', keep='first').reset_index(drop=True)

        # ----------------------------- MODO PA -----------------------------
        else:
            # Regra: se soma PI == 0, trata o pacote como PA
            # Importante: NÃO filtrar somente PA.
            # Mantém o ecossistema completo (PA + PI zerado) para análise/visibilidade.
            bd_ECO = bd_LTP_MES_DIM.copy()

            # Hora alvo e PCS/H usados no motor são os de PA
            col_hora_alvo = "NEC_ATEND_HR"

            # Se quiser enxergar explicitamente o modo aplicado no pacote (debug)
            bd_ECO["MODO_CORTE"] = "PA"

        # ======================================================================================
        # 6) MOTOR DE CORTE (aplica mesma lógica para PI e PA)
        # ======================================================================================

        # ----------------------------------------- Realizar Rateio ---------------------------------------------

        # Colunas que serão ajustadas pelo rateio proporcional
        cols_rateio = ["C_ARR_HR","C_AT_HR","PV_HR","PV_PROX_HR","ES_HR","DIF_LM_HR","DIF_EMB_HR"]

        # Capturar modo direto
        modo = bd_ECO["MODO_CORTE"].iloc[0]

        # Definir máscara
        if modo == "PI":
            mask_rateio = bd_ECO["RASTREABILIDADE"].notna() & bd_ECO["RASTREABILIDADE"].ne("")
        else:
            mask_rateio = slice(None)  # todas as linhas

        # LOG: garantir colunas auxiliares para snapshots ANT_/DEP_
        bd_ECO = garantir_colunas_log(bd_ECO, COLS_ECO_ANTES + COLS_ECO_DEPOIS)

        # Base usada para calcular a proporção entre as colunas
        rateio_base = bd_ECO.loc[mask_rateio, cols_rateio]

        # Soma da base de rateio por linha
        soma_base_rateio = rateio_base.sum(axis=1)

        # Total que será redistribuído
        total_rateio = bd_ECO.loc[mask_rateio, col_hora_alvo].where(soma_base_rateio > 0, 0)

        # Aplicar rateio
        bd_ECO.loc[mask_rateio, cols_rateio] = (
            rateio_base
            .div(soma_base_rateio, axis=0)
            .mul(total_rateio, axis=0)
            .fillna(0)
        )

        # Snapshot
        bd_ECO_snapshot = bd_ECO.copy(deep=True)

        # --------------------------------------- Fim Realizar Rateio -------------------------------------------
        # Na Matriz de cortes fazer um loop para cada coluna seguindo a sequencia da propria matriz, somente para colunas que contem hora

        # Função que cria a matriz de cortes, e sequencia de cortes que deve ser respeitada
        bd_MAT_LOG_CORTES = matriz_logica_cortes_horas(bd_ECO, col_rec_fer)

        # Identifica colunas da matriz no formato "MESMA_REG|COLUNA_HR"
        cols_matriz_log = [c for c in bd_MAT_LOG_CORTES.columns if "|" in c]

        # LOG: sequência interna da matlog dentro da iteração da chave
        seq_matlog = 0

        # Loop sobre as colunas da matriz logica de cortes, seguindo a sequencia definida na própria matriz logica
        for col_matriz in cols_matriz_log:

            seq_matlog += 1

            mesma_reg_alvo, col_hr_alvo = col_matriz.split("|", 1)
            val_tot_hor_col_mat_log = float(bd_MAT_LOG_CORTES[col_matriz].sum())
            objetivo_corte_hr_evento = float(val_hor_objetivo_mat_corte)

            # Snapshot do ANTES REAL do evento (já considerando o saldo do evento anterior)
            bd_ECO_ANTES = bd_ECO.copy(deep=True)

            tipo_corte = "SEM_CORTE"
            motivo_sem_corte = None
            val_hor_para_cortar = 0.0
            corte_aplicado = False

            col_ltp_alvo = MAP_HR_TO_LTP.get(col_hr_alvo)

            # Se não houver mais horas para cortar, registra a visita e sai do loop
            if val_hor_objetivo_mat_corte <= 0:
                motivo_sem_corte = "OBJETIVO_ZERADO"
                bd_ECO_DEPOIS = bd_ECO.copy(deep=True)
                id_evento_corte += 1

                metadata_evento = {
                    "ID_EVENTO_CORTE": id_evento_corte,
                    "OBJETIVO_CORTE_HR": objetivo_corte_hr_evento,
                    "TIPO_CORTE": tipo_corte,
                    "MOTIVO_SEM_CORTE": motivo_sem_corte,
                    "CORTE_APLICADO": corte_aplicado,
                    "MES_REF_EVENTO": mes_ref,
                    "DIMENSAO_CORTE": dimensao_corte,
                    "CAMPO_DIMENSAO_CORTE": col_rec_fer,
                    "VALOR_DIMENSAO_CORTE": chave_dim,
                    "UNID_PROD_EVENTO": unid_prod,
                    "MODO_EVENTO": modo,
                    "ITERACAO": iteracao_chave,
                    "SEQ_MATLOG": seq_matlog,
                    "COLUNA_MATLOG": col_matriz,
                    "MESMA_REG_ALVO": mesma_reg_alvo,
                    "COL_HR_ALVO": col_hr_alvo,
                    "COL_LTP_ALVO": col_ltp_alvo,
                    "TOTAL_BD_MAT_LOG_CORTES": val_tot_hor_col_mat_log,
                    "HORAS_CORTADAS_EVENTO": float(val_hor_para_cortar),
                    "OBJETIVO_RESTANTE_POS_EVENTO": float(val_hor_objetivo_mat_corte),
                }

                registrar_evento_log(
                    logs_buffer=logs_cortes,
                    bd_eco_antes=bd_ECO_ANTES,
                    bd_eco_depois=bd_ECO_DEPOIS,
                    metadata_evento=metadata_evento
                )
                break

            # Sem saldo nesta coluna da matlog
            if val_tot_hor_col_mat_log <= 0:
                motivo_sem_corte = "SEM_SALDO_NA_COLUNA"
                bd_ECO_DEPOIS = bd_ECO.copy(deep=True)
                id_evento_corte += 1

                metadata_evento = {
                    "ID_EVENTO_CORTE": id_evento_corte,
                    "OBJETIVO_CORTE_HR": objetivo_corte_hr_evento,
                    "TIPO_CORTE": tipo_corte,
                    "MOTIVO_SEM_CORTE": motivo_sem_corte,
                    "CORTE_APLICADO": corte_aplicado,
                    "MES_REF_EVENTO": mes_ref,
                    "DIMENSAO_CORTE": dimensao_corte,
                    "CAMPO_DIMENSAO_CORTE": col_rec_fer,
                    "VALOR_DIMENSAO_CORTE": chave_dim,
                    "UNID_PROD_EVENTO": unid_prod,
                    "MODO_EVENTO": modo,
                    "ITERACAO": iteracao_chave,
                    "SEQ_MATLOG": seq_matlog,
                    "COLUNA_MATLOG": col_matriz,
                    "MESMA_REG_ALVO": mesma_reg_alvo,
                    "COL_HR_ALVO": col_hr_alvo,
                    "COL_LTP_ALVO": col_ltp_alvo,
                    "TOTAL_BD_MAT_LOG_CORTES": val_tot_hor_col_mat_log,
                    "HORAS_CORTADAS_EVENTO": float(val_hor_para_cortar),
                    "OBJETIVO_RESTANTE_POS_EVENTO": float(val_hor_objetivo_mat_corte),
                }

                registrar_evento_log(
                    logs_buffer=logs_cortes,
                    bd_eco_antes=bd_ECO_ANTES,
                    bd_eco_depois=bd_ECO_DEPOIS,
                    metadata_evento=metadata_evento
                )
                continue

            # Se o valor total da matriz logica for maior que 0, ou seja, houver horas para cortar, aplicar o corte seguindo a sequencia da matriz logica
            if val_tot_hor_col_mat_log <= val_hor_objetivo_mat_corte:

                # Calcular valor possivel de corte para a coluna da matriz, considerando o total da coluna e as horas restantes para corte
                val_hor_para_cortar = val_tot_hor_col_mat_log

                # Na bd_ECO, zerar a coluna col_hr_alvo, para as linhas que se encaixam na mesma_reg_alvo, e atualizar o valor objetivo de corte subtraindo o valor cortado
                mask_corte_total = bd_ECO["MESMA_REG"] == mesma_reg_alvo
                horas_antes_corte = bd_ECO.loc[mask_corte_total, col_hr_alvo].copy()

                bd_ECO.loc[mask_corte_total, col_hr_alvo] = 0
                val_hor_objetivo_mat_corte -= val_hor_para_cortar

                if col_ltp_alvo is not None:
                    # LOG: controle antes específico do LTP saldo prev
                    if col_ltp_alvo == "LTP_SALDO_PREV_PCS":
                        bd_ECO.loc[mask_corte_total, "LTP_SALDO_PREV_PCS_ANTES"] = bd_ECO.loc[mask_corte_total, col_ltp_alvo].values

                    # LOG: em corte total, RATEIO=1 para linhas efetivamente cortadas
                    bd_ECO.loc[mask_corte_total, "RATEIO"] = np.where(
                        bd_ECO.loc[mask_corte_total, col_ltp_alvo] > 0, 1.0, 0.0
                    )
                    bd_ECO.loc[mask_corte_total, "CORTE_HOR"] = horas_antes_corte.values
                    bd_ECO.loc[mask_corte_total, "CORTE_PCS"] = bd_ECO.loc[mask_corte_total, col_ltp_alvo].values
                    bd_ECO.loc[mask_corte_total, "SALDO_PCS"] = 0.0
                    bd_ECO.loc[mask_corte_total, col_ltp_alvo] = 0

                    if col_ltp_alvo == "LTP_SALDO_PREV_PCS":
                        bd_ECO.loc[mask_corte_total, "LTP_SALDO_PREV_PCS_DEPOIS"] = bd_ECO.loc[mask_corte_total, col_ltp_alvo].values

                tipo_corte = "CORTE_TOTAL"
                corte_aplicado = True

            else:
                # Se o valor total da matriz logica for maior que o valor objetivo de corte, aplicar rateio proporcional para cortar o valor objetivo de corte, atualizar o valor objetivo de corte subtraindo o valor cortado, e ir para próxima coluna da matriz logica

                # Calcular valor possivel de corte para a coluna da matriz, considerando o total da coluna e as horas restantes para corte
                val_hor_para_cortar = val_hor_objetivo_mat_corte

                # Gerar uma cópia da bd_ECO com nome bd_ECO_RATEIO, aplicando o filtro de mesma_reg_alvo, para evitar modificar a bd_ECO original durante o processo de rateio proporcional
                bd_ECO_RATEIO = bd_ECO.loc[bd_ECO["MESMA_REG"] == mesma_reg_alvo].copy()

                # Criando a coluna RATEIO com valor 0 para todas as linhas inicialmente
                bd_ECO_RATEIO["RATEIO"] = 0.0

                # Variavel com total da coluna col_hr_alvo
                val_tot_col_hr_alvo = bd_ECO_RATEIO[col_hr_alvo].sum()

                # Na coluna RATEIO, calcular a proporção de cada linha em relação ao total da coluna col_hr_alvo, para as linhas onde o valor da coluna col_hr_alvo seja maior que 0, caso contrário, manter o valor de RATEIO como 0 para evitar divisão por zero
                bd_ECO_RATEIO["RATEIO"] = np.where(
                    bd_ECO_RATEIO[col_hr_alvo] > 0,
                    bd_ECO_RATEIO[col_hr_alvo] / val_tot_col_hr_alvo,
                    0
                )

                # Criar coluna CORTE_HOR na bd_ECO_RATEIO, calculando as horas cortadas por linha, multiplicando o RATEIO pelo valor objetivo de corte (val_hor_para_cortar), e garantindo que o corte nunca passe do HR disponível no balde (col_hr_alvo)
                bd_ECO_RATEIO["CORTE_HOR"] = bd_ECO_RATEIO["RATEIO"] * val_hor_para_cortar
                bd_ECO_RATEIO["CORTE_HOR"] = np.minimum(bd_ECO_RATEIO["CORTE_HOR"], bd_ECO_RATEIO[col_hr_alvo])

                # Criar a coluna CORTE_PCS na ECO_RATEIO calculando a conversão de CORTE_HOR para PCS
                if col_ltp_alvo is not None:
                    if modo == "PI":
                        bd_ECO_RATEIO["CORTE_PCS"] = np.where(
                            bd_ECO_RATEIO["CORTE_HOR"].eq(0),
                            bd_ECO_RATEIO[col_ltp_alvo],
                            (bd_ECO_RATEIO["CORTE_HOR"] * bd_ECO_RATEIO["PCS_HORA_PI"]) / bd_ECO_RATEIO["FATOR_ESTRUTURAL"]
                        )
                    else:
                        bd_ECO_RATEIO["CORTE_PCS"] = np.where(
                            bd_ECO_RATEIO["CORTE_HOR"].eq(0),
                            bd_ECO_RATEIO[col_ltp_alvo],
                            bd_ECO_RATEIO["CORTE_HOR"] * bd_ECO_RATEIO["PCS_HORA"]
                        )

                    # Criar coluna SALDO_PCS na bd_ECO_RATEIO
                    bd_ECO_RATEIO["SALDO_PCS"] = bd_ECO_RATEIO[col_ltp_alvo] - bd_ECO_RATEIO["CORTE_PCS"]

                    # Quando o rateio = 0, faz uma busca do minimo do saldo_pçs > zero, comparando ID_PROD_UNID_FAT, informando o resultado no saldo_pcs
                    menor_saldo_por_id = (
                        bd_ECO_RATEIO[bd_ECO_RATEIO["SALDO_PCS"] > 0]
                        .groupby("ID_PROD_UNID_FAT")["SALDO_PCS"]
                        .min()
                    )

                    mask = bd_ECO_RATEIO["RATEIO"] == 0

                    bd_ECO_RATEIO.loc[mask, "SALDO_PCS"] = (
                        bd_ECO_RATEIO.loc[mask, "ID_PROD_UNID_FAT"]
                        .map(menor_saldo_por_id)
                        .fillna(bd_ECO_RATEIO.loc[mask, "SALDO_PCS"])
                    )

                    # Atualizar a coluna LTP_ correspondente na bd_ECO_RATEIO, utilizando o mapping para propagar o SALDO_PCS para a coluna estrutural LTP correspondente
                    if col_ltp_alvo == "LTP_SALDO_PREV_PCS":
                        bd_ECO_RATEIO["LTP_SALDO_PREV_PCS_ANTES"] = bd_ECO_RATEIO[col_ltp_alvo]

                    bd_ECO_RATEIO[col_ltp_alvo] = bd_ECO_RATEIO["SALDO_PCS"]

                    if col_ltp_alvo == "LTP_SALDO_PREV_PCS":
                        bd_ECO_RATEIO["LTP_SALDO_PREV_PCS_DEPOIS"] = bd_ECO_RATEIO[col_ltp_alvo]

                    # Levar as colunas RATEIO, CORTE_HOR, CORTE_PCS, SALDO_PCS para a bd_ECO, utilizando o IND como índice técnico para garantir que as atualizações sejam aplicadas corretamente mesmo que haja linhas repetidas no ecossistema
                    colunas_adicionais = ["RATEIO", "CORTE_HOR", "CORTE_PCS", "SALDO_PCS", "LTP_SALDO_PREV_PCS_ANTES", "LTP_SALDO_PREV_PCS_DEPOIS"]
                    for col in colunas_adicionais:
                        if col in bd_ECO_RATEIO.columns:
                            map_col = bd_ECO_RATEIO[["IND", col]].drop_duplicates(subset=["IND"], keep="last").set_index("IND")[col]
                            mask_col = bd_ECO["IND"].isin(map_col.index)
                            bd_ECO.loc[mask_col, col] = bd_ECO.loc[mask_col, "IND"].map(map_col).values

                    # Gerar uma cópia da coluna LTP_alvo da bd_ECO, com o nome col_ltp_alvo_antes, para manter o valor antes do corte para análises posteriores
                    col_ltp_alvo_antes = col_ltp_alvo + "_ANTES"
                    bd_ECO[col_ltp_alvo_antes] = bd_ECO[col_ltp_alvo]

                    # Atualizar bd_ECO com os dados das colunas LTP correspondentes da bd_ECO_RATEIO
                    map_atualizacao = bd_ECO_RATEIO[["IND", col_ltp_alvo]].drop_duplicates(subset=["IND"], keep="last").set_index("IND")[col_ltp_alvo]
                    mask_atualizacao = bd_ECO["IND"].isin(map_atualizacao.index)
                    bd_ECO.loc[mask_atualizacao, col_ltp_alvo] = bd_ECO.loc[mask_atualizacao, "IND"].map(map_atualizacao).values

                    # Gerar uma cópia da coluna LTP_alvo da bd_ECO, com o nome col_ltp_alvo_depois
                    col_ltp_alvo_depois = col_ltp_alvo + "_DEPOIS"
                    bd_ECO[col_ltp_alvo_depois] = bd_ECO[col_ltp_alvo]

                # Atualizar a coluna col_hr_alvo na bd_ECO com base na coluna col_hr_alvo da bd_ECO_RATEIO
                map_hr_atualizacao = bd_ECO_RATEIO[["IND", col_hr_alvo]].drop_duplicates(subset=["IND"], keep="last").set_index("IND")[col_hr_alvo]
                mask_hr_atualizacao = bd_ECO["IND"].isin(map_hr_atualizacao.index)
                bd_ECO.loc[mask_hr_atualizacao, col_hr_alvo] = bd_ECO.loc[mask_hr_atualizacao, "IND"].map(map_hr_atualizacao).values

                # Atualizar o valor objetivo de corte subtraindo o valor cortado da coluna da matriz logica, para controlar o loop de cortes
                val_hor_objetivo_mat_corte -= val_hor_para_cortar

                tipo_corte = "RATEIO"
                corte_aplicado = True

            id_evento_corte += 1
            bd_ECO_DEPOIS = bd_ECO.copy(deep=True)

            metadata_evento = {
                "ID_EVENTO_CORTE": id_evento_corte,
                "OBJETIVO_CORTE_HR": objetivo_corte_hr_evento,
                "TIPO_CORTE": tipo_corte,
                "MOTIVO_SEM_CORTE": motivo_sem_corte,
                "CORTE_APLICADO": corte_aplicado,
                "MES_REF_EVENTO": mes_ref,
                "DIMENSAO_CORTE": dimensao_corte,
                "CAMPO_DIMENSAO_CORTE": col_rec_fer,
                "VALOR_DIMENSAO_CORTE": chave_dim,
                "UNID_PROD_EVENTO": unid_prod,
                "MODO_EVENTO": modo,
                "ITERACAO": iteracao_chave,
                "SEQ_MATLOG": seq_matlog,
                "COLUNA_MATLOG": col_matriz,
                "MESMA_REG_ALVO": mesma_reg_alvo,
                "COL_HR_ALVO": col_hr_alvo,
                "COL_LTP_ALVO": col_ltp_alvo,
                "TOTAL_BD_MAT_LOG_CORTES": float(val_tot_hor_col_mat_log),
                "HORAS_CORTADAS_EVENTO": float(val_hor_para_cortar),
                "OBJETIVO_RESTANTE_POS_EVENTO": float(val_hor_objetivo_mat_corte),
            }

            registrar_evento_log(
                logs_buffer=logs_cortes,
                bd_eco_antes=bd_ECO_ANTES,
                bd_eco_depois=bd_ECO_DEPOIS,
                metadata_evento=metadata_evento
            )

            # Atualizar as colunas _LTP da bd_LTP_MES, com base nas mesmas colunas _LTP da bd_ECO
            eco_idx = bd_ECO.set_index("IND")

            for col_ltp in [
                "LTP_CART_ARR_MES_ANT",
                "LTP_CART_MES_ATUAL",
                "LTP_SALDO_PREV_PCS",
                "LTP_SALDO_PREV_PROX_MES_PCS",
                "LTP_EST_SEG_PCS"
            ]:
                bd_LTP_MES[col_ltp] = bd_LTP_MES["IND"].map(eco_idx[col_ltp]).fillna(bd_LTP_MES[col_ltp])

        # FIXME: Gerar snapshot da bd_MAT_HOR_CORTES após os cortes para análise de impacto (comparar com snapshot antes dos cortes)
        bd_MAT_HOR_CORTES_antes_de_atualizar = bd_MAT_HOR_CORTES.copy(deep=True)

        # ======================================================================================
        # 7) Pós-corte: recalcular modelo completo, reexplodir estrutura, recalcular necessidades e recriar base de cortes
        # ======================================================================================
        bd_LTP_MES = calc_nec_pcs_hr(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
        bd_LTP_MES, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
        bd_LTP_MES = calcular_demais_campos(bd_LTP_MES)

        # ******************# Explosão da Estrutura e necessidade de componentes (pós-corte)
        bd_ltp_estrutura_explodida = explodir_estrutura_ltp(bd_estrutura_filtrada, bd_LTP_MES)
        bd_nec_comp_expl = calcular_explosao_necessidades(bd_ltp_estrutura_explodida, bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
        bd_LTP_MES = atualizar_ltp_comp_nec_pcs(bd_LTP_MES, bd_nec_comp_expl)

        # ******************# Recalculando NEC_PCS e demais campos, após explosão componentes (pós-corte)
        bd_LTP_MES = calc_nec_pcs_hr(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
        bd_LTP_MES, tab_HOR_REC, tab_HOR_FER = calcular_distrib_capacidade(bd_LTP_MES, lote_min_flag, multiplo_emb_flag)
        bd_LTP_MES = calcular_demais_campos(bd_LTP_MES)

        # ******************# Decomposição para precisão de corte na nova rodada
        bd_LTP_MES = calcular_decomposicao_nec_pcs_para_precisao_corte(bd_LTP_MES)

        # ******************# Criar controle de iterações
        tab_iteracoes = pd.DataFrame(
            [
                {
                    "MES_REF": mes_ref,
                    col_rec_fer: k[0],
                    "UNID_PROD": k[1],
                    "QTD_ITERACOES": v
                }
                for k, v in contador.items()
            ]
        )

        tab_iteracoes = tab_iteracoes.sort_values(
            by=["QTD_ITERACOES", col_rec_fer, "UNID_PROD"],
            ascending=[False, True, True]
        ).reset_index(drop=True)


# LOG: Gerar parquet de auditoria dos cortes
if logs_cortes:
    df_log_cortes = pd.concat(logs_cortes, ignore_index=True)

    cols_ordenacao = [
        col for col in [
            "MES_REF_EVENTO",
            "DIMENSAO_CORTE",
            "VALOR_DIMENSAO_CORTE",
            "UNID_PROD_EVENTO",
            "ITERACAO",
            "SEQ_MATLOG",
            "ID_EVENTO_CORTE",
            "LOG_IND",
        ] if col in df_log_cortes.columns
    ]

    if cols_ordenacao:
        df_log_cortes = df_log_cortes.sort_values(cols_ordenacao).reset_index(drop=True)

    # gerar em excel para análise
    df_log_cortes.to_excel(
        pasta_output / f"log_cortes_{mes_ref.strftime('%Y_%m')}.xlsx",
        index=False
    )

DEBUG PI: chave_dim=01812 unid_prod=Matriz
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (18, 94)
qtd linhas com COD_INSUMO preenchido = 12
DEBUG PI: chave_dim=EX504 unid_prod=Nordeste
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (16, 94)
qtd linhas com COD_INSUMO preenchido = 5
DEBUG PI: chave_dim=EX504 unid_prod=Nordeste
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (10, 94)
qtd linhas com COD_INSUMO preenchido = 1
DEBUG PI: chave_dim=EX504 unid_prod=Nordeste
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (10, 94)
qtd linhas com COD_INSUMO preenchido = 1
DEBUG PI: chave_dim=01401 unid_prod=Matriz
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (39, 94)
qtd linhas com COD_INSUMO preenchido = 13
DEBUG PI: chave_dim=01801 unid_prod=Matriz
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (29, 94)
qtd linhas com COD_INSUMO preenchido = 17
DEBUG PI: chave_dim=00518 unid_prod=Matriz
tem COD_INSUMO no final?  True
shape bd_ECO_PI = (20, 94)
qtd linhas com COD_INSUMO preenchido =

In [9]:
# Exportar para Excel

dataframes_para_exportar = {
    "bd_LTP_MES_snapshot": bd_LTP_MES_snapshot,
    "bd_LTP_MES": bd_LTP_MES,
    "bd_LTP_MES_DIM": bd_LTP_MES_DIM
}

for nome in dataframes_para_exportar:
    df = globals()[nome]
    caminho_arquivo = pasta_output / f"{nome}.xlsx"
    df.to_excel(caminho_arquivo, index=False)
    print(f"✅ Exportado: {caminho_arquivo}")
    
timer.finalizar()
print("🎯 Processo concluído com sucesso!")

✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES_snapshot.xlsx
✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES.xlsx
✅ Exportado: c:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT\bd_LTP_MES_DIM.xlsx
Tempo total de processamento: 4 min 37.3 s
🎯 Processo concluído com sucesso!


In [10]:
# CRIAR UM LOG DE CORTES, PARA SABER COMO FOI PROCESSADO E SE FOI PROCESSADO

# Ver quais Bibliotecas devo excluir conforme indicado por Lucas
# Não recomendamos a instalação pois possui vulnerabilidades não corrigidas
# colorama
# numpy
# prompt toolkit
# python-dateutil
